In [1]:
from truckscenes import TruckScenes
trucksc = TruckScenes('v1.0-trainval', '/home/cx/server3/man-truckscenes/', True)
trucksc.list_scenes()
terminal_scene_list = []
count = 0
for scene in trucksc.scene:
    # if 'terminal' in scene['description']:
        # print(scene['description'])
    if count > 101 and count < 110:
        terminal_scene_list.append(scene)
    count += 1

for index, terminal_scene in enumerate(terminal_scene_list):
    print(index, terminal_scene['description'])

Loading truckscenes tables for version v1.0-trainval...
11 attribute,
18 calibrated_sensor,
27 category,
1200243 ego_motion_cabin,
1200183 ego_motion_chassis,
1201993 ego_pose,
43769 instance,
23902 sample,
819536 sample_annotation,
2586264 sample_data,
598 scene,
18 sensor,
4 visibility,
Done loading in 57.812 seconds.
Reverse indexing ...
Done reverse indexing in 8.2 seconds.
Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[23-08-17 06:10:20] scene-e1dbd50ff4694a46b4997f4dec87f667-5: length: 19.50s, #anns: 858, desc: weather.clear;area.highway;daytime.morning;season.s...
[23-08-17 06:15:00] scene-e1dbd50ff4694a46b4997f4dec87f667-19: length: 19.50s, #anns: 1370, desc: weather.clear;area.highway;daytime.morning;season.s...
[23-08-17 06:31:02] scene-b502fc44d4324e1cb0349f8fa46d4014-7: length: 19.50s, #anns: 1206, desc: weather.clear;area.highway;daytime.morning;season

In [ ]:
from truckscenes.utils.data_classes import LidarPointCloud
from pyquaternion import Quaternion
from truckscenes.utils.geometry_utils import transform_matrix
import numpy as np
import os.path as osp
import json
import math
from functools import reduce

LIDAR_OUTPUT_PATH = '/home/cx/server3/TruckScenes/v1.0-trainval/sample'
LABEL_OUTPUT_PATH = '/home/cx/server3/TruckScenes/v1.0-trainval/label'
LIDAR_REF = "LIDAR_LEFT"
USE_REF_COORD = True


def yaw_to_quaternion(yaw_rad):
    """将yaw角转换为四元数
    Args:
        yaw_rad: 弧度制偏航角
    Returns:
        [x, y, z, w] 格式四元数
    """
    cy = math.cos(yaw_rad * 0.5)
    sz = math.sin(yaw_rad * 0.5)
    return [0.0, 0.0, sz, cy]   

for index, terminal_scene in enumerate(terminal_scene_list):
    first_sample_token = terminal_scene['first_sample_token']
    last_sample_token = terminal_scene['first_sample_token']
    current_sample_token = first_sample_token
    while True:
        if current_sample_token == '':
            break
        # if (current_sample_token == last_sample_token and current_sample_token != first_sample_token):
        #     break
        sample = trucksc.get('sample', current_sample_token)
        # print(sample.keys())
        timestamp = sample['timestamp']
        ego_pose = trucksc.getclosest('ego_pose', timestamp)
        # print(ego_pose)
        
        # sample_data = trucksc.get('sample_data', sample['sample_token'])

        ref_lidar_sd_token = sample['data'][LIDAR_REF]
        ref_lidar_sample_data = trucksc.get('sample_data', ref_lidar_sd_token)
        ref_cs = trucksc.get('calibrated_sensor', ref_lidar_sample_data['calibrated_sensor_token'])
        ref_pose_rec = trucksc.get('ego_pose', ref_lidar_sample_data['ego_pose_token'])
        ref_from_car = transform_matrix(ref_cs['translation'],
                                        Quaternion(ref_cs['rotation']),
                                        inverse=True)

        # Homogeneous transformation matrix from global to _current_ ego car frame.
        car_from_global = transform_matrix(ref_pose_rec['translation'],
                                           Quaternion(ref_pose_rec['rotation']),
                                           inverse=True)
 

        point_data_list = []
        for key, val in sample['data'].items():
            if 'LIDAR' in key:
                # print(key, val)
                sensor_token = val
                lidar_sample_data = trucksc.get('sample_data', sensor_token)
                # print(lidar_sample_data)
                cs_record = trucksc.get('calibrated_sensor', lidar_sample_data['calibrated_sensor_token'])
                # get vehicle frame lidar
                pcl_path = osp.join(trucksc.dataroot, lidar_sample_data['filename'])
                pc = LidarPointCloud.from_file(pcl_path)
                
                current_pose_rec = trucksc.get('ego_pose', lidar_sample_data['ego_pose_token'])

                global_from_car = transform_matrix(current_pose_rec['translation'],
                                               Quaternion(current_pose_rec['rotation']),
                                               inverse=False)

                # Homogeneous transformation matrix from sensor coordinate frame to ego car frame.
                # current_cs_rec = trucksc.get('calibrated_sensor',
                #                         cs_record['calibrated_sensor_token'])
                car_from_current = transform_matrix(cs_record['translation'],
                                                    Quaternion(cs_record['rotation']),
                                                    inverse=False)

                # Fuse four transformation matrices into one and perform transform.
                if USE_REF_COORD:
                    trans_matrix = reduce(np.dot, [ref_from_car, car_from_global,
                                                global_from_car, car_from_current])
                else:
                    trans_matrix = reduce(np.dot, [car_from_global,
                                                global_from_car, car_from_current])
                pc.transform(trans_matrix)

                # pc.rotate(Quaternion(cs_record['rotation']).rotation_matrix)
                # pc.translate(np.array(cs_record['translation']))


                # poserecord = trucksc.get('ego_pose', lidar_sample_data['ego_pose_token'])
                # pc.rotate(Quaternion(poserecord['rotation']).rotation_matrix)
                # pc.translate(np.array(poserecord['translation']))


                # pc.translate(-np.array(ego_pose['translation']))
                # pc.rotate(Quaternion(ego_pose['rotation']).rotation_matrix.T)
                point_data_list.append(pc.points)


                # print(pc.points.shape)
        if (len(point_data_list) > 0):
            points = np.concatenate(point_data_list, axis=1).T
            # print(points)
            out_file = osp.join(LIDAR_OUTPUT_PATH, '{}.bin'.format(str(timestamp)))
            print(out_file)
            points.tofile(out_file)

        # print(sample_data)
        annos = sample['anns']
        out_file = osp.join(LABEL_OUTPUT_PATH, '{}.json'.format(str(timestamp)))
        label_data = []
        with open(out_file, 'w', encoding='utf-8') as f:
            for anno in annos:
                anno_data = trucksc.get('sample_annotation', anno)
                cls_name = anno_data['category_name']
                # print(anno_data)
                box = trucksc.get_box(anno)
                box.translate(-np.array(ref_pose_rec['translation']))
                box.rotate(Quaternion(ref_pose_rec['rotation']).inverse)
                if USE_REF_COORD:
                    box.translate(-np.array(ref_cs['translation']))
                    box.rotate(Quaternion(ref_cs['rotation']).inverse)

                box_data = {
                    "track_id": 1, 
                    "label": cls_name, 
                    "subtype": cls_name, 
                    "xyz": [box.center[0], box.center[1], box.center[2]], 
                    "lwh": [box.wlh[1], box.wlh[0], box.wlh[2]], 
                    "rotation": box.orientation.yaw_pitch_roll[0], 
                    "num_lidar_pts": 10
                }
                label_data.append(box_data)
            json.dump(label_data, f)
        current_sample_token = sample['next']
    # break

In [ ]:
terminal_scene_list
